# Práctica 1: Adquisición y representación de imágenes de productos

Análisis de fotografías propias: resolución, profundidad de color, RGB, gris, blanco y negro, HSV, histogramas y segmentación. Ejecuta las celdas en orden. Para analizar otra foto, cambia `NOMBRE_IMAGEN` y vuelve a ejecutar desde esa celda.

## 1. Librerías y fotografías

Si faltan librerías, ejecuta en la terminal `python -m pip install -r requirements.txt`. Las fotografías deben estar en `imagenes`, junto a este cuaderno.

In [ ]:
from pathlib import Path
import csv
import cv2
import numpy as np
import matplotlib.pyplot as plt

BASE = Path.cwd()
ENTRADA = BASE / "imagenes"
SALIDA = BASE / "resultados" / "cuaderno"
SALIDA.mkdir(parents=True, exist_ok=True)
extensiones = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
archivos = sorted(p for p in ENTRADA.iterdir() if p.suffix.lower() in extensiones)
if not archivos:
    raise ValueError("Agrega fotografías a la carpeta imagenes.")
print(f"Fotografías disponibles: {len(archivos)}")
for ruta in archivos:
    print(ruta.name)

## 2. Adquisición y características

Se incluyen nueve fotografías de dos productos, leche condensada y Milo, desde diferentes vistas. Completar el dispositivo, la distancia y la iluminación utilizados en la captura. Las medidas siguientes corresponden a las matrices cargadas por OpenCV en tres canales de 8 bits; no certifican la profundidad original del archivo.

In [ ]:
filas = []
figura, ejes = plt.subplots((len(archivos) + 2) // 3, 3, figsize=(12, 4 * ((len(archivos) + 2) // 3)))
for eje in np.asarray(ejes).flat:
    eje.axis("off")
for eje, ruta in zip(np.asarray(ejes).flat, archivos):
    foto = cv2.imdecode(np.fromfile(str(ruta), dtype=np.uint8), cv2.IMREAD_COLOR)
    if foto is None:
        raise ValueError(f"No se pudo leer {ruta.name}")
    alto, ancho, canales = foto.shape
    filas.append({"archivo": ruta.name, "ancho_px": ancho, "alto_px": alto,
                  "megapixeles": round(ancho * alto / 1_000_000, 3),
                  "canales": canales, "bits_por_canal": foto.dtype.itemsize * 8,
                  "bits_por_pixel": foto.dtype.itemsize * 8 * canales,
                  "memoria_bytes": foto.nbytes, "archivo_bytes": ruta.stat().st_size})
    eje.imshow(cv2.cvtColor(foto, cv2.COLOR_BGR2RGB))
    eje.set_title(f"{ruta.name}\n{ancho} × {alto} px")
figura.tight_layout()
plt.show()
for fila in filas:
    print(f"{fila['archivo']}: {fila['megapixeles']} MP | {fila['bits_por_pixel']} bits/píxel | matriz: {fila['memoria_bytes']} bytes | archivo: {fila['archivo_bytes']} bytes")

## 3. Selección del producto

Cambia el nombre por cualquiera de la lista. Se inicia con Milo y el color verde. Los colores disponibles son rojo, amarillo, verde y azul.

In [ ]:
NOMBRE_IMAGEN = "miloFront.jpeg"
COLOR = "verde"
UMBRAL = 127

ruta = ENTRADA / NOMBRE_IMAGEN
imagen = cv2.imdecode(np.fromfile(str(ruta), dtype=np.uint8), cv2.IMREAD_COLOR)
if imagen is None:
    raise ValueError(f"No se pudo leer {NOMBRE_IMAGEN}")
rgb = cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB)
alto, ancho, canales = imagen.shape
print(f"Resolución: {ancho} × {alto} píxeles")
print(f"Profundidad cargada: {imagen.dtype.itemsize * 8} bits por canal")
print(f"Canales: {canales}")
print(f"Tipo de dato: {imagen.dtype}")
plt.figure(figsize=(6, 6))
plt.imshow(rgb)
plt.title(NOMBRE_IMAGEN)
plt.axis("off")
plt.show()

## 4. RGB, escala de grises y blanco y negro

OpenCV lee en BGR; se convierte a RGB para mostrar los colores correctamente. Gris combina aproximadamente 0.299R + 0.587G + 0.114B. La binarización asigna blanco a valores mayores que el umbral y negro al resto. La matriz binaria sigue ocupando 8 bits por píxel, aunque solo utiliza dos valores.

In [ ]:
gris = cv2.cvtColor(imagen, cv2.COLOR_BGR2GRAY)
_, binaria = cv2.threshold(gris, UMBRAL, 255, cv2.THRESH_BINARY)
figura, ejes = plt.subplots(1, 3, figsize=(13, 5))
for eje, vista, titulo in zip(ejes, [rgb, gris, binaria], ["RGB", "Escala de grises", f"Blanco y negro: umbral {UMBRAL}"]):
    eje.imshow(vista, cmap="gray", vmin=0, vmax=255)
    eje.set_title(titulo)
    eje.axis("off")
figura.tight_layout()
figura.savefig(SALIDA / f"{ruta.stem}_representaciones.png", dpi=150)
plt.show()

## 5. Espacio HSV

H representa el tono (0–179), S la saturación (0–255) y V el brillo (0–255) en OpenCV para imágenes de 8 bits. Se visualizan los canales separados en gris: las zonas claras indican valores altos del canal.

In [ ]:
hsv = cv2.cvtColor(imagen, cv2.COLOR_BGR2HSV)
h, s, v = cv2.split(hsv)
figura, ejes = plt.subplots(1, 3, figsize=(13, 5))
for eje, canal, titulo, limite in zip(ejes, [h, s, v], ["H: tono", "S: saturación", "V: brillo"], [179, 255, 255]):
    vista = eje.imshow(canal, cmap="gray", vmin=0, vmax=limite)
    figura.colorbar(vista, ax=eje, fraction=0.046)
    eje.set_title(titulo)
    eje.axis("off")
figura.tight_layout()
figura.savefig(SALIDA / f"{ruta.stem}_hsv.png", dpi=150)
plt.show()

## 6. Histogramas básicos

Cada histograma cuenta los píxeles por intensidad o tono. En gris, los valores cercanos a cero son oscuros y los cercanos a 255 son claros.

In [ ]:
figura, ejes = plt.subplots(1, 3, figsize=(14, 4))
ejes[0].plot(cv2.calcHist([gris], [0], None, [256], [0, 256]), color="black")
ejes[0].set_title("Gris")
for indice, color, nombre in zip(range(3), ["red", "green", "blue"], ["R", "G", "B"]):
    histograma = cv2.calcHist([rgb], [indice], None, [256], [0, 256])
    ejes[1].plot(histograma, color=color, label=nombre)
ejes[1].set_title("RGB")
ejes[1].legend()
ejes[2].plot(cv2.calcHist([hsv], [0], None, [180], [0, 180]), color="purple")
ejes[2].set_title("Tono H")
for indice, eje in enumerate(ejes):
    eje.set_xlabel("Tono" if indice == 2 else "Intensidad")
    eje.set_ylabel("Número de píxeles")
    eje.set_xlim(0, 179 if indice == 2 else 255)
figura.tight_layout()
figura.savefig(SALIDA / f"{ruta.stem}_histogramas.png", dpi=150)
plt.show()

## 7. Segmentación por color

Los rangos son valores iniciales ajustables. La máscara marca en blanco los píxeles seleccionados. El rojo utiliza dos intervalos por estar en ambos extremos de H. El porcentaje indica superficie seleccionada de la fotografía; no mide cuántos productos se reconocieron.

In [ ]:
rangos = {
    "rojo": [((0, 80, 50), (10, 255, 255)), ((170, 80, 50), (179, 255, 255))],
    "amarillo": [((20, 80, 50), (35, 255, 255))],
    "verde": [((36, 60, 40), (85, 255, 255))],
    "azul": [((90, 80, 50), (130, 255, 255))]
}
mascara = np.zeros(gris.shape, dtype=np.uint8)
for inferior, superior in rangos[COLOR]:
    parcial = cv2.inRange(hsv, np.array(inferior, dtype=np.uint8), np.array(superior, dtype=np.uint8))
    mascara = cv2.bitwise_or(mascara, parcial)
segmentada = cv2.bitwise_and(rgb, rgb, mask=mascara)
porcentaje = cv2.countNonZero(mascara) / mascara.size * 100
figura, ejes = plt.subplots(1, 3, figsize=(13, 5))
for eje, vista, titulo in zip(ejes, [rgb, mascara, segmentada], ["Original", f"Máscara: {COLOR}", "Segmentación"]):
    eje.imshow(vista, cmap="gray", vmin=0, vmax=255)
    eje.set_title(titulo)
    eje.axis("off")
figura.tight_layout()
figura.savefig(SALIDA / f"{ruta.stem}_segmentacion_{COLOR}.png", dpi=150)
plt.show()
print(f"Porcentaje de píxeles seleccionados: {porcentaje:.2f}%")

## 8. Guardar datos

Las figuras y las tablas se guardan en `resultados/cuaderno`. La tabla de características contiene todas las fotografías; la segmentación corresponde a la seleccionada.

In [ ]:
with (SALIDA / "caracteristicas.csv").open("w", newline="", encoding="utf-8-sig") as archivo:
    escritor = csv.DictWriter(archivo, fieldnames=filas[0].keys())
    escritor.writeheader()
    escritor.writerows(filas)
with (SALIDA / f"{ruta.stem}_segmentacion_{COLOR}.csv").open("w", newline="", encoding="utf-8-sig") as archivo:
    escritor = csv.writer(archivo)
    escritor.writerow(["imagen", "color", "umbral_binario", "porcentaje_seleccionado"])
    escritor.writerow([NOMBRE_IMAGEN, COLOR, UMBRAL, round(porcentaje, 2)])
print(f"Resultados guardados en {SALIDA}")

## 9. Análisis y conclusiones

Completar después de revisar las salidas:

1. ¿Qué fotografía tiene mayor resolución? ¿Cómo se relaciona su memoria con el tamaño del archivo?
2. ¿Qué detalles se pierden al pasar a gris y luego a blanco y negro?
3. ¿Qué picos de los histogramas corresponden al producto y al fondo?
4. ¿La máscara selecciona el producto completo o solo zonas del color buscado?
5. ¿Qué errores aparecen por sombras, reflejos o colores del fondo?
6. Si se dispone de dos capturas con igual encuadre y distinta iluminación, comparar usando los mismos rangos HSV. Las distintas vistas por sí solas no permiten aislar el efecto de la iluminación.

**Conclusiones:** pendientes del análisis de los resultados. HSV separa tono y brillo, pero la segmentación sigue dependiendo de los rangos y las condiciones de captura.

## Referencias

- [LearnOpenCV: Color spaces in OpenCV](https://learnopencv.com/color-spaces-in-opencv-cpp-python/)
- [PyImageSearch: OpenCV Color Spaces](https://pyimagesearch.com/2021/04/28/opencv-color-spaces-cv2-cvtcolor/)
- [PyImageSearch: OpenCV Image Histograms](https://pyimagesearch.com/2021/04/28/opencv-image-histograms-cv2-calchist/)
- [OpenCV: Changing Colorspaces](https://docs.opencv.org/4.x/df/d9d/tutorial_py_colorspaces.html)